## **Import Section**
Importing all the necessary libraries

In [1]:
import pandas as pd
import numpy as np
import linopy as lp
import xarray as xr
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.graph_objects as go
from joblib import Parallel, delayed
import multiprocessing


## **Inputting the Data**
To begin our modelling, we have to input the necessary data. For this case, we need the spot market prices and the solar pv capacity factors for each location. The values that were extracted are from real large-scale PV power plants to try to imitate a real life scenario as much as possible. There are four locations, each location is either located in the north, south, east, or west of Germany to get a full scope of the effect of the location on the investment.

In [2]:
input_file = pd.read_csv(r'Operations/Input_File.csv', sep=';',  decimal=',')

spot_prices     = (input_file['Spot-Prices (€/MWh)'].values) / 1000   # €/MWh -> €/kWh 

capacity_factors = {
    "Inden_CF":            input_file.iloc[:, 3].values,
    "Weesow_Willmersdorf": input_file.iloc[:, 4].values,
    "Lauingen":            input_file.iloc[:, 5].values,
    "Eggebek":             input_file.iloc[:, 6].values}

## **Defining the index sets**
Here the coordinate set was built that were used throughout the model

In [3]:
time = pd.Index(range(len(input_file)))
region    = list(capacity_factors.keys())
commodity = ["electricity"]
component = ["capacity factors", "spot prices"]

## **Array set**
Here an empty 4-dimensional array was created and indexed by time, region, commodity and component.

In [4]:
selling = xr.DataArray(
    np.zeros((len(time), len(region), len(commodity), len(component))), # filled with zeros at first
    dims=("time", "region", "commodity", "component"),
    coords={
        "time": time,
        "region": region,
        "commodity": commodity,
        "component": component
    }
)

## **Populating the Array**
The array gets populated, where the capacity factor gets assigned to each location and the spot prices are introduced.

In [5]:
selling.loc[dict(region="Inden_CF",
                 commodity="electricity",
                 component="capacity factors")] = capacity_factors["Inden_CF"]

selling.loc[dict(region="Weesow_Willmersdorf",
                 commodity="electricity",
                 component="capacity factors")] = capacity_factors["Weesow_Willmersdorf"]

selling.loc[dict(region="Lauingen",
                 commodity="electricity",
                 component="capacity factors")] = capacity_factors["Lauingen"]

selling.loc[dict(region="Eggebek",
                 commodity="electricity",
                 component="capacity factors")] = capacity_factors["Eggebek"]

selling.loc[dict(region="Inden_CF",
                 commodity="electricity",
                 component="spot prices")] = spot_prices

selling.loc[dict(region="Weesow_Willmersdorf",
                 commodity="electricity",
                 component="spot prices")] = spot_prices

selling.loc[dict(region="Lauingen",
                 commodity="electricity",
                 component="spot prices")] = spot_prices

selling.loc[dict(region="Eggebek",
                 commodity="electricity",
                 component="spot prices")] = spot_prices

## **Parameters**
Here are the input variables needed to run the code:
1. The discount rate of PV and Battery are assumed as 2.25% as the key interest rate by the european central bank, respectively.
2. The lifetime of PV and Battery are 27.1 and 15 years, respectively.
3. The annuity factors are calculated with the interest rate and the lifetime, respectively
4. The maintenance factors for PV and Battery are assumed as 1% and 2.5% of the investment costs per year
5. CAPEX of PV and Battery are 497 €/kW and 387.5 €/kWh, respectively.
6. To define the battery's power rating, a fixed power-to-capacity ratio of 1/2.32 (≈0.43C) was applied to the installed energy capacity.
7. The Depth of Discharge (DoD) was taken to be 80 %, max is 90% and minimum is 10%.
8. The budget is a limit we put onto the system in which how much we are allowed to spend. 

In [6]:
discount_rate_pv   = 0.0225 #ecb key interest rate
discount_rate_batt = 0.0225 #ecb key interest rate
lifetime_pv        = 27.1
lifetime_batt      = 15

annuity_factor_pv   = (discount_rate_pv   * (1 + discount_rate_pv)**lifetime_pv)   / ((1 + discount_rate_pv)**lifetime_pv   - 1)
annuity_factor_batt = (discount_rate_batt * (1 + discount_rate_batt)**lifetime_batt) / ((1 + discount_rate_batt)**lifetime_batt - 1)

maintenance_factor_pv = 0.01    # percent of invest cost
maintenance_factor_batt = 0.025 # percent of invest cost

capex_pv_per_kw     = 497 # €/kW  — raw upfront cost, used for budget ### source: Ghadim et al. 2025 min 323	mean 497	max 1114
capex_batt_per_kwh  = 387.5 # €/kWh — raw upfront cost, used for budget ### source: Figgener et al. 2023 LSS 465 - 310

battery_efficiency  = 0.86  #roundtrip
ratio_cap_vs_pwr    = 1/2.32
budget              = 1_000_000   # € upfront cash available

soc_min_frac = 0.10   # DoD floor: never discharge below 10% of installed capacity
soc_max_frac = 0.90   # DoD ceiling: never charge above 90% of installed capacity

## **The Optimization Model**
The module 'linopy' was used for the optimization model, where two **investment decision variables** `pv_cap` and `batt_cap`, one per region, was used. The battery's power rating (`power_cap`) was defined by the `batt_cap` multiplied by the fixed C-rate ratio. Then six **operational variables** are introduced (continuous, indexed by time and region):
- PV generation
- battery charge/discharge
- state of charge (SOC)
- and electricity sold to the grid from PV and from the battery separately.

In [7]:
m = lp.Model()

# 1. Establish explicitly named multi-dimensional coordinates
variable_coords = {"time": time, "region": region}

# 2. Add investment capacities (Forced to Integers using integer=True)
pv_cap    = m.add_variables(lower=0, coords={"region": region}, name="pv_cap") 
batt_cap  = m.add_variables(lower=0, coords={"region": region}, name="batt_cap") 

# 3. Calculate your power capacity scaling factor mapping
power_cap = batt_cap * ratio_cap_vs_pwr

# 4. Continuous operational variables (tracked across time and region)
pv_gen         = m.add_variables(lower=0, coords=variable_coords, name="pv_gen")
batt_charge    = m.add_variables(lower=0, coords=variable_coords, name="batt_charge")
batt_discharge = m.add_variables(lower=0, coords=variable_coords, name="batt_discharge")
soc            = m.add_variables(lower=0, coords=variable_coords, name="soc")
sell_from_pv   = m.add_variables(lower=0, coords=variable_coords, name="sell_from_pv")
sell_from_batt = m.add_variables(lower=0, coords=variable_coords, name="sell_from_batt")

## **Constraints**
- This cell first extracts two slices from the selling array: cf_xr, the hourly capacity factor for each region, and prices_xr, the hourly spot price.
- `pv_gen <= cf_xr * pv_cap`: in any hour, the electricity actually generated cannot exceed the installed PV capacity multiplied by that hour's capacity factor. Because this is written as a "less than or equal to" rather than an equality, the model is allowed to curtail, in other words, generate less than the maximum available, if that turns out to be economically preferable (for example, if selling that power would only be sellable at a negative price).
- `pv_gen == batt_charge + sell_from_pv`: the energy balance is split between two destinations: every unit of electricity generated must either be sold directly to the grid or routed into the battery for storage.
- `sell_from_batt == batt_discharge`: the efficiency is only applied on the charging process. Therefore the discharged energy is completly sold
- `soc >= soc_min_frac * batt_cap` & `soc <= soc_max_frac * batt_cap`: Two constraints enforce depth-of-discharge limits where the state of charge must stay between a lower bound (10% of installed capacity) and an upper bound (90% of installed capacity). This mirrors standard battery operating practice, where fully draining or fully filling a battery accelerates degradation, so a buffer is kept at both ends.
- `batt_charge <= power_cap` & `batt_charge <= power_cap`: Two more constraints impose charge and discharge power limits, tied to the battery's power rating (itself derived from its energy capacity via the fixed power-to-capacity ratio). These act like an inverter or C-rate limit.
- `batt_discharge(t) <= soc.sel(t-1) - soc_min_frac * batt_cap` & `batt_charge(t) <= (soc_max_frac * batt_cap - soc.sel(t-1)) / battery_efficiency`: These check against what's actually available or free right now, based on the previous hour's state of charge net of the depth-of-discharge floor and ceiling. In other words, the battery can't discharge more energy than it actually holds above the 10% floor, and it can't charge with more energy than there's room for below the 90% ceiling — even if the power limit alone would technically allow it.
- `capex_pv_per_kw * pv_cap + capex_batt_per_kwh * batt_cap <= budget`: A single budget constraint then caps the total upfront capital spent on PV and battery capacity combined.
- `soc(t) == soc(t-1) + batt_charge(t) * battery_efficiency - batt_discharge(t)` & `soc(t=0) == soc_min_frac * batt_cap`: The SOC dynamics constraint links each hour's state of charge to the previous hour's, adding whatever was charged (net of efficiency losses) and subtracting whatever was discharged. The SOC initialization constraint is a special case of the same logic applied at the very first timestep, where there is no "previous hour" to reference, so the battery is instead assumed to start at the depth-of-discharge floor before any charging or discharging in that first hour is applied.
- No non-negativity constraint is given since these were already classified at the variable declaration as lower=0.

In [8]:
cf_xr     = selling.sel(commodity="electricity", component="capacity factors")
prices_xr = selling.sel(commodity="electricity", component="spot prices")

# 1. PV Generation Ceiling (curtailment allowed)
m.add_constraints(pv_gen <= cf_xr * pv_cap, name="pv_gen_max")

# 2. PV Energy Split (Flow Balance)
m.add_constraints(pv_gen == batt_charge + sell_from_pv, name="pv_balance")

# 3. Discharge Efficiency (applied once, on the way out to the grid)
m.add_constraints(sell_from_batt == batt_discharge, name="batt_sell")

# 4. Depth of Discharge — SOC bounded between 10% and 90% of installed capacity
m.add_constraints(soc >= soc_min_frac * batt_cap, name="soc_min_dod")
m.add_constraints(soc <= soc_max_frac * batt_cap, name="soc_max_dod")

# 5. Operational Power Constraints (Inverter Limits)
m.add_constraints(batt_charge <= power_cap, name="charge_power_limit")
m.add_constraints(batt_discharge <= power_cap, name="discharge_power_limit")

# 6. Available Energy/Room Slices (relative to the DoD floor/ceiling, not 0/batt_cap)
m.add_constraints(
    batt_discharge.sel(time=time[1:]) <= soc.sel(time=time[:-1]) - soc_min_frac * batt_cap,
    name="discharge_volume_limit"
)
m.add_constraints(
    batt_charge.sel(time=time[1:]) <= (soc_max_frac * batt_cap - soc.sel(time=time[:-1])) / battery_efficiency,
    name="charge_volume_limit"
)

# 7. Budget Cap
m.add_constraints(capex_pv_per_kw * pv_cap + capex_batt_per_kwh * batt_cap <= budget, name="budget")

# 8. SOC Dynamics
m.add_constraints(
    soc.sel(time=time[1:]) == soc.sel(time=time[:-1])
        + batt_charge.sel(time=time[1:]) * battery_efficiency
        - batt_discharge.sel(time=time[1:]),
    name="soc_dynamics"
)

# 9. SOC Initialization — battery starts pre-charged at the DoD floor (physically realistic)
m.add_constraints(
    soc.sel(time=0) == soc_min_frac * batt_cap,
    name="soc_init"
)

Constraint `soc_init` [region: 4]:
----------------------------------
[Inden_CF]: +1 soc[0, Inden_CF] - 0.1 batt_cap[Inden_CF]                                  = -0.0
[Weesow_Willmersdorf]: +1 soc[0, Weesow_Willmersdorf] - 0.1 batt_cap[Weesow_Willmersdorf] = -0.0
[Lauingen]: +1 soc[0, Lauingen] - 0.1 batt_cap[Lauingen]                                  = -0.0
[Eggebek]: +1 soc[0, Eggebek] - 0.1 batt_cap[Eggebek]                                     = -0.0

## **Objective Function**
The objective function is to maximize the profit, in where annual revenue as spot price × total electricity sold (from PV and battery, summed over time and region) is subtracted from the annualized CAPEX as installed capacity × unit cost × annuity factor for each technology.

In [9]:
annual_revenue = (prices_xr * (sell_from_pv + sell_from_batt)).sum()

annual_capex = (capex_pv_per_kw     * annuity_factor_pv   * pv_cap
               + capex_batt_per_kwh * annuity_factor_batt * batt_cap).sum("region")

annual_maintenance = (capex_pv_per_kw * pv_cap * maintenance_factor_pv + capex_batt_per_kwh * batt_cap * maintenance_factor_batt)

m.add_objective(annual_revenue - annual_capex - annual_maintenance, sense="max")
#m.solve()
m.solve(solver_name="highs")

Writing continuous variables.: 100%|██████████| 8/8 [00:00<00:00, 216.82it/s]


('ok', 'optimal')

## **Exporting each region's time series results**
For each region, an hourly `DataFrame` of spot price, PV generation, battery charge/discharge, SOC, and grid sales from PV/battery is produced, then the `DataFrame` is saved to a region-specific CSV file.

In [10]:
# Create a dictionary to hold a separate DataFrame for each region
regional_dfs = {}

for r in region:
    # Extract the time-series variables for this specific region
    # Also extract the aligned spot prices for this region to see market context
    df = pd.DataFrame({
        "Market_Spot_Price_EUR_per_kWh": prices_xr.sel(region=r).to_series(),
        "PV_Generation_kW":              pv_gen.solution.sel(region=r).to_series(),
        "Battery_Charge_kW":             batt_charge.solution.sel(region=r).to_series(),
        "Battery_Discharge_kW":          batt_discharge.solution.sel(region=r).to_series(),
        "State_of_Charge_kWh":           soc.solution.sel(region=r).to_series(),
        "Sell_from_PV_kW":               sell_from_pv.solution.sel(region=r).to_series(),
        "Sell_from_Batt_kW":             sell_from_batt.solution.sel(region=r).to_series()
    }, index=time)
    
    # Save to our dictionary
    regional_dfs[r] = df
    
    # Define a clean filename dynamically named after the region
    filename = f"results_{r}.csv"
    
    # Optional: Write metadata headers at the very top of the file before the spreadsheet rows
    with open(filename, 'w') as f:
        f.write(f"# Region:;{r}\n")
        f.write(f"# Optimized PV Capacity (kW):;{pv_cap.solution.sel(region=r).item():.2f}\n")
        f.write(f"# Optimized Battery Capacity (kWh):;{batt_cap.solution.sel(region=r).item():.2f}\n")
        
    # Append the main hourly time-series spreadsheet rows (including Spot Prices) right below the headers
    df.to_csv(filename, mode='a', sep=';', index_label="Time", decimal = ',')
    print(f"Successfully saved results with price context for {r} to {filename}")

Successfully saved results with price context for Inden_CF to results_Inden_CF.csv
Successfully saved results with price context for Weesow_Willmersdorf to results_Weesow_Willmersdorf.csv
Successfully saved results with price context for Lauingen to results_Lauingen.csv
Successfully saved results with price context for Eggebek to results_Eggebek.csv


## **Summary table of the results**
For each region, the optimized PV/battery capacity, totals generation and battery discharge is printed out, the PV and battery revenue and total CAPEX spent is calculated, then displayed to a `DataFrame` and saves it to `regional_summary_report.csv`.

In [11]:
# Create an empty list to store each region's summary dictionary
summary_data = []

for r in region:
    # 1. Extract the scalar optimized capacities for this region
    # .item() extracts the native Python float/int from a 0-dimensional array
    pv_installed   = float(pv_cap.solution.sel(region=r).item())
    batt_installed = float(batt_cap.solution.sel(region=r).item())
    
    # 2. Sum the hourly operational series for this region
    total_pv_gen          = float(pv_gen.solution.sel(region=r).sum().item())
    total_batt_discharged = float(batt_discharge.solution.sel(region=r).sum().item())
    
    # 3. Calculate revenues using the vectorized array calculations for this region
    # prices_xr matches your spot_prices but is already aligned to the time coordinates
    revenue_pv   = float((prices_xr.sel(region=r) * sell_from_pv.solution.sel(region=r)).sum().item())
    revenue_batt = float((prices_xr.sel(region=r) * sell_from_batt.solution.sel(region=r)).sum().item())
    total_revenue = revenue_pv + revenue_batt

    # Annualisierte CAPEX (Annuität) für diese Region
    annual_capex_r = (capex_pv_per_kw   * annuity_factor_pv   * pv_installed
                     + capex_batt_per_kwh * annuity_factor_batt * batt_installed)

    # Jährliche Wartungskosten für diese Region
    annual_maintenance_r = (capex_pv_per_kw   * pv_installed   * maintenance_factor_pv
                           + capex_batt_per_kwh * batt_installed * maintenance_factor_batt)

    total_profit = total_revenue - annual_capex_r - annual_maintenance_r
        
    # 4. Calculate CAPEX spent
    capex_spent = (capex_pv_per_kw * pv_installed) + (capex_batt_per_kwh * batt_installed)
    
    # Store everything in a dictionary for this specific region
    summary_data.append({
        "Region": r,
        "PV_Installed_kW": round(pv_installed, 2),
        "Batt_Installed_kWh": round(batt_installed, 2),
        "Total_PV_Gen_kWh": round(total_pv_gen, 2),
        "Total_Batt_Discharged_kWh": round(total_batt_discharged, 2),
        "Revenue_PV_EUR": round(revenue_pv, 2),
        "Revenue_Batt_EUR": round(revenue_batt, 2),
        "Total_Revenue_EUR": round(total_revenue, 2),
        "CAPEX_Spent_EUR": capex_spent,
        "total_profit": round(total_profit, 2)
    })

# Convert the list of summaries into a single final DataFrame
df_summary = pd.DataFrame(summary_data).set_index("Region")

# Display the summary table
print("\n=== REGIONAL RESULTS SUMMARY ===")
display(df_summary)

# Optional: Save the summary report to a separate CSV file
df_summary.to_csv("regional_summary_report.csv", sep=';')


=== REGIONAL RESULTS SUMMARY ===


,PV_Installed_kW,Batt_Installed_kWh,Total_PV_Gen_kWh,Total_Batt_Discharged_kWh,Revenue_PV_EUR,Revenue_Batt_EUR,Total_Revenue_EUR,CAPEX_Spent_EUR,total_profit
Region,,,,,,,,,
Inden_CF,2012.07,-0.0,1868670.02,0.0,136004.56,0.0,136004.56,1000000.0,76316.79
Weesow_Willmersdorf,2012.07,-0.0,1798724.35,0.0,132900.93,0.0,132900.93,1000000.0,73213.15
Lauingen,2012.07,-0.0,1966573.44,0.0,147329.64,0.0,147329.64,1000000.0,87641.86
Eggebek,2012.07,-0.0,1706444.67,0.0,121338.83,0.0,121338.83,1000000.0,61651.06


## **Sensitivity Analysis and Visualization**

1. **Reusable model function** with identical constraints and objective function but only for one region that is used for the sensitivity analysis
2. **Sensitivity analysis for CAPEX** (PV & Battery) as a Heatmap.
3. **Time series plots** for the SOC, PV energy production and Spot market prices.

### 1. Reusable model function for sensitivity analysis

The function builds the same model as before but includes just one region at the time and adds Capex and Discount values as input arguments. Therefore it can be used for the sensitivity analysis.

In [12]:
def solve_single_region_model(
    region_name,
    capex_pv_per_kw=capex_pv_per_kw,
    capex_batt_per_kwh=capex_batt_per_kwh,
    discount_rate_pv=discount_rate_pv,
    discount_rate_batt=discount_rate_batt,
    budget=budget,
    lifetime_pv=lifetime_pv,
    lifetime_batt=lifetime_batt,
    maintenance_factor_pv=maintenance_factor_pv,
    maintenance_factor_batt=maintenance_factor_batt,
    battery_efficiency=battery_efficiency,
    ratio_cap_vs_pwr=ratio_cap_vs_pwr,
    soc_min_frac=soc_min_frac,
    soc_max_frac=soc_max_frac,
    solver_name="highs",
    io_api="direct",          
    return_timeseries=True,   
):

    annuity_factor_pv_local = (
        discount_rate_pv * (1 + discount_rate_pv) ** lifetime_pv
    ) / ((1 + discount_rate_pv) ** lifetime_pv - 1)
    annuity_factor_batt_local = (
        discount_rate_batt * (1 + discount_rate_batt) ** lifetime_batt
    ) / ((1 + discount_rate_batt) ** lifetime_batt - 1)

    cf_local = cf_xr.sel(region=region_name)
    prices_local = prices_xr.sel(region=region_name)

    m_local = lp.Model()

    pv_cap_l = m_local.add_variables(lower=0, name="pv_cap", integer=True)
    batt_cap_l = m_local.add_variables(lower=0, name="batt_cap", integer=True)
    power_cap_l = batt_cap_l * ratio_cap_vs_pwr

    pv_gen_l = m_local.add_variables(lower=0, coords={"time": time}, name="pv_gen")
    batt_charge_l = m_local.add_variables(lower=0, coords={"time": time}, name="batt_charge")
    batt_discharge_l = m_local.add_variables(lower=0, coords={"time": time}, name="batt_discharge")
    soc_l = m_local.add_variables(lower=0, coords={"time": time}, name="soc")
    sell_from_pv_l = m_local.add_variables(lower=0, coords={"time": time}, name="sell_from_pv")
    sell_from_batt_l = m_local.add_variables(lower=0, coords={"time": time}, name="sell_from_batt")

    m_local.add_constraints(pv_gen_l <= cf_local * pv_cap_l, name="pv_gen_max")
    m_local.add_constraints(pv_gen_l == batt_charge_l + sell_from_pv_l, name="pv_balance")
    m_local.add_constraints(sell_from_batt_l == batt_discharge_l, name="batt_sell")
    m_local.add_constraints(soc_l >= soc_min_frac * batt_cap_l, name="soc_min_dod")
    m_local.add_constraints(soc_l <= soc_max_frac * batt_cap_l, name="soc_max_dod")
    m_local.add_constraints(batt_charge_l <= power_cap_l, name="charge_power_limit")
    m_local.add_constraints(batt_discharge_l <= power_cap_l, name="discharge_power_limit")
    m_local.add_constraints(
        batt_discharge_l.sel(time=time[1:]) <= soc_l.sel(time=time[:-1]) - soc_min_frac * batt_cap_l,
        name="discharge_volume_limit",
    )
    m_local.add_constraints(
        batt_charge_l.sel(time=time[1:])
        <= (soc_max_frac * batt_cap_l - soc_l.sel(time=time[:-1])) / battery_efficiency,
        name="charge_volume_limit",
    )
    m_local.add_constraints(
        capex_pv_per_kw * pv_cap_l + capex_batt_per_kwh * batt_cap_l <= budget, name="budget"
    )
    m_local.add_constraints(
        soc_l.sel(time=time[1:])
        == soc_l.sel(time=time[:-1])
        + batt_charge_l.sel(time=time[1:]) * battery_efficiency
        - batt_discharge_l.sel(time=time[1:]),
        name="soc_dynamics",
    )
    m_local.add_constraints(soc_l.sel(time=0) == soc_min_frac * batt_cap_l, name="soc_init")

    annual_revenue_l = (prices_local * (sell_from_pv_l + sell_from_batt_l)).sum()
    annual_capex_l = (
        capex_pv_per_kw * annuity_factor_pv_local * pv_cap_l
        + capex_batt_per_kwh * annuity_factor_batt_local * batt_cap_l
    )
    annual_maintenance_l = (
        capex_pv_per_kw * pv_cap_l * maintenance_factor_pv
        + capex_batt_per_kwh * batt_cap_l * maintenance_factor_batt
    )

    m_local.add_objective(annual_revenue_l - annual_capex_l - annual_maintenance_l, sense="max")
    status, condition = m_local.solve(solver_name=solver_name, io_api=io_api)

    if status != "ok":
        result = {
            "region": region_name,
            "capex_pv": capex_pv_per_kw,
            "capex_batt": capex_batt_per_kwh,
            "discount_rate_pv": discount_rate_pv,
            "discount_rate_batt": discount_rate_batt,
            "objective": np.nan,
            "pv_cap": np.nan,
            "batt_cap": np.nan,
        }
    else:
        result = {
            "region": region_name,
            "capex_pv": capex_pv_per_kw,
            "capex_batt": capex_batt_per_kwh,
            "discount_rate_pv": discount_rate_pv,
            "discount_rate_batt": discount_rate_batt,
            "objective": float(m_local.objective.value),
            "pv_cap": float(pv_cap_l.solution.item()),
            "batt_cap": float(batt_cap_l.solution.item()),
        }

    if return_timeseries:
        result["pv_gen"] = pv_gen_l.solution if status == "ok" else None
        result["soc"] = soc_l.solution if status == "ok" else None
        result["sell_from_pv"] = sell_from_pv_l.solution if status == "ok" else None
        result["sell_from_batt"] = sell_from_batt_l.solution if status == "ok" else None

    return result


### 2. Sensitivity analysis for CAPEX of PV & Battery

`SENSITIVITY_REGION` can be set to the regions (`"Inden_CF"`, `"Weesow_Willmersdorf"`, `"Lauingen"`, `"Eggebek"`). A range can be set for the Capex with `capex_pv_range` and `capex_batt_range`. The model is calculated for every combination and the profit is saved in a matrix.

In [13]:
# Parallesizing the computing power 

N_JOBS = -1  # -1 use all available cores
print(f"Available CPU-Cores: {multiprocessing.cpu_count()}")

Available CPU-Cores: 16


### Results Heatmap

The heatmap is displayed at first only for one region and aftwerwards for all four regions. 

In [14]:
# Manually selection for which region the sensitivity analysis should be calculated
SENSITIVITY_REGION = region[2]   # z.B. "Inden_CF", "Weesow_Willmersdorf", "Lauingen" oder "Eggebek"

# Range for the Capex can be adapted here
capex_pv_range   = np.linspace(323, 1114, 10)
capex_batt_range = np.linspace(50, 465, 10)

param_grid = [(i, j, c_batt, c_pv)
              for i, c_batt in enumerate(capex_batt_range)
              for j, c_pv in enumerate(capex_pv_range)]


#Calculating the model
def _capex_job(c_pv, c_batt):
    result = solve_single_region_model(
        region_name=SENSITIVITY_REGION,
        capex_pv_per_kw=c_pv,
        capex_batt_per_kwh=c_batt,
        discount_rate_pv=discount_rate_pv,
        discount_rate_batt=discount_rate_batt,
        io_api="direct",
        return_timeseries=False,
    )
    
    return result["objective"], result["pv_cap"], result["batt_cap"]

results_flat = Parallel(n_jobs=N_JOBS)(
    delayed(_capex_job)(c_pv, c_batt) for (_, _, c_batt, c_pv) in param_grid
)

#Saving the Results in three seperate matrices: Profit, installed PV-capacity and installed battery capacity
profit_matrix_capex   = np.zeros((len(capex_batt_range), len(capex_pv_range)))
pv_cap_matrix_capex   = np.zeros((len(capex_batt_range), len(capex_pv_range)))
batt_cap_matrix_capex = np.zeros((len(capex_batt_range), len(capex_pv_range)))

for (i, j, _, _), (objective, pv_cap_val, batt_cap_val) in zip(param_grid, results_flat):
    profit_matrix_capex[i, j]   = objective
    pv_cap_matrix_capex[i, j]   = pv_cap_val
    batt_cap_matrix_capex[i, j] = batt_cap_val

In [ ]:
#plotting the results in heatmaps

fig_capex_heatmaps = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        f"PV Capacity - Region: {SENSITIVITY_REGION}",
        f"Battery Capacity - Region: {SENSITIVITY_REGION}",
        f"Annual Profit - Region: {SENSITIVITY_REGION}",
    ),
    horizontal_spacing=0.15,
)

x_labels = [f"{v:.0f}" for v in capex_pv_range]
y_labels = [f"{v:.0f}" for v in capex_batt_range]

# 1. PV capacity heatmap
fig_capex_heatmaps.add_trace(
    go.Heatmap(
        z=pv_cap_matrix_capex,
        x=x_labels, y=y_labels,
        colorscale="RdYlGn",
        text=pv_cap_matrix_capex,
        texttemplate="%{text:.0f}",
        colorbar=dict(title="Capacity (kW)", x=0.27, len=1.0),
    ),
    row=1, col=1,
)

# 2. Battery capacity heatmap
fig_capex_heatmaps.add_trace(
    go.Heatmap(
        z=batt_cap_matrix_capex,
        x=x_labels, y=y_labels,
        colorscale="RdYlGn",
        text=batt_cap_matrix_capex,
        texttemplate="%{text:.0f}",
        colorbar=dict(title="Capacity (kWh)", x=0.635, len=1.0),
    ),
    row=1, col=2,
)

# 3. Profit heatmap
fig_capex_heatmaps.add_trace(
    go.Heatmap(
        z=profit_matrix_capex,
        x=x_labels, y=y_labels,
        colorscale="RdYlGn",
        text=profit_matrix_capex,
        texttemplate="%{text:.0f}",
        colorbar=dict(title="Profit (Euro)", x=1.0, len=1.0),
    ),
    row=1, col=3,
)

# Shared axis labels + reversed y-axis on all three subplots
for col in (1, 2, 3):
    fig_capex_heatmaps.update_xaxes(title_text="CAPEX PV (Euro/kW)", row=1, col=col)
    fig_capex_heatmaps.update_yaxes(title_text="CAPEX Battery (Euro/kWh)", autorange="reversed", row=1, col=col)

fig_capex_heatmaps.update_layout(
    title_text=f"Sensitivity Analysis: CAPEX PV & Battery - Region: {SENSITIVITY_REGION}",
    height=500,
    width=1500,
)

fig_capex_heatmaps.show()

### Heatmaps for all Regions

Here the same Heatmaps from before are generated but as a comparison for all regions

In [16]:
# Range for the Capex can be adapted here
capex_pv_range   = np.linspace(323, 1114, 10)
capex_batt_range = np.linspace(50, 465, 10)

param_grid = [(i, j, c_batt, c_pv)
              for i, c_batt in enumerate(capex_batt_range)
              for j, c_pv in enumerate(capex_pv_range)]

def _capex_job(region_name, c_pv, c_batt):
    result = solve_single_region_model(
        region_name=region_name,
        capex_pv_per_kw=c_pv,
        capex_batt_per_kwh=c_batt,
        discount_rate_pv=discount_rate_pv,
        discount_rate_batt=discount_rate_batt,
        io_api="direct",
        return_timeseries=False,
    )
    
    return result["objective"], result["pv_cap"], result["batt_cap"]


profit_matrix_capex_by_region   = {}
pv_cap_matrix_capex_by_region   = {}
batt_cap_matrix_capex_by_region = {}

for r in region:
    results_flat = Parallel(n_jobs=N_JOBS)(
        delayed(_capex_job)(r, c_pv, c_batt) for (_, _, c_batt, c_pv) in param_grid
    )

    profit_matrix   = np.zeros((len(capex_batt_range), len(capex_pv_range)))
    pv_cap_matrix   = np.zeros((len(capex_batt_range), len(capex_pv_range)))
    batt_cap_matrix = np.zeros((len(capex_batt_range), len(capex_pv_range)))

    for (i, j, _, _), (objective, pv_cap_val, batt_cap_val) in zip(param_grid, results_flat):
        profit_matrix[i, j]   = objective
        pv_cap_matrix[i, j]   = pv_cap_val
        batt_cap_matrix[i, j] = batt_cap_val

    profit_matrix_capex_by_region[r]   = profit_matrix
    pv_cap_matrix_capex_by_region[r]   = pv_cap_matrix
    batt_cap_matrix_capex_by_region[r] = batt_cap_matrix

    print(f"CAPEX-Sensitivitaetsanalyse for region '{r}' done ")

CAPEX-Sensitivitaetsanalyse for region 'Inden_CF' done 
CAPEX-Sensitivitaetsanalyse for region 'Weesow_Willmersdorf' done 


KeyboardInterrupt: 

In [ ]:
n_regions = len(region)

fig_capex_heatmaps = make_subplots(
    rows=n_regions, cols=3,
    subplot_titles=["PV Capacity", "Battery Capacity", "Annual Profit"] + [""] * (3 * (n_regions - 1)),
    row_titles=list(region),
    horizontal_spacing=0.12,
    vertical_spacing=0.06,
)

x_labels = [f"{v:.0f}" for v in capex_pv_range]
y_labels = [f"{v:.0f}" for v in capex_batt_range]

for row_idx, r in enumerate(region, start=1):
    # 1. PV capacity heatmap
    fig_capex_heatmaps.add_trace(
        go.Heatmap(
            z=pv_cap_matrix_capex_by_region[r],
            x=x_labels, y=y_labels,
            coloraxis="coloraxis1",
        ),
        row=row_idx, col=1,
    )
    # 2. Battery capacity heatmap
    fig_capex_heatmaps.add_trace(
        go.Heatmap(
            z=batt_cap_matrix_capex_by_region[r],
            x=x_labels, y=y_labels,
            coloraxis="coloraxis2",
        ),
        row=row_idx, col=2,
    )
    # 3. Profit heatmap
    fig_capex_heatmaps.add_trace(
        go.Heatmap(
            z=profit_matrix_capex_by_region[r],
            x=x_labels, y=y_labels,
            coloraxis="coloraxis3",
        ),
        row=row_idx, col=3,
    )

for col in (1, 2, 3):
    fig_capex_heatmaps.update_xaxes(title_text="CAPEX PV (Euro/kW)", row=n_regions, col=col)
for row_idx in range(1, n_regions + 1):
    fig_capex_heatmaps.update_yaxes(title_text="CAPEX Batt. (Euro/kWh)", autorange="reversed", row=row_idx, col=1)
    fig_capex_heatmaps.update_yaxes(autorange="reversed", row=row_idx, col=2)
    fig_capex_heatmaps.update_yaxes(autorange="reversed", row=row_idx, col=3)


fig_capex_heatmaps.update_layout(
    title_text="CAPEX Sensitivity - PV & Battery Capacity / Profit by Region",
    coloraxis1=dict(colorscale="RdYlGn", colorbar=dict(title="kW",   x=0.27,  len=1.0)),
    coloraxis2=dict(colorscale="RdYlGn", colorbar=dict(title="kWh",  x=0.635, len=1.0)),
    coloraxis3=dict(colorscale="RdYlGn", colorbar=dict(title="Euro", x=1.0,   len=1.0)),
    height=220 * n_regions,
    width=1200,
)

fig_capex_heatmaps.show()

## 3. Time series plots
The Spot market price, the State of Charge of the battery and the PV-generation are plotted for the whole time series. 
By the default case battery CAPEX of 387.5€/kWh, the optimization model does not use any battery capacity


In [ ]:
PLOT_START_DATE = "2025-01-01 00:00"
timestamps = pd.date_range(start=PLOT_START_DATE, periods=len(time), freq="h")

plot_frames = []
for r in region:
    plot_frames.append(pd.DataFrame({
        "timestamp": timestamps,
        "region": r,
        "Spot_price_EUR_per_kWh": prices_xr.sel(region=r).values,
        "PV_generation_kW": pv_gen.solution.sel(region=r).values,
        "State_of_Charge_SOC_kWh": soc.solution.sel(region=r).values,
        "Sell_from_PV_kW": sell_from_pv.solution.sel(region=r).values,
        "Sell_from_Battery_kW": sell_from_batt.solution.sel(region=r).values,
    }))

df_timeseries = pd.concat(plot_frames, ignore_index=True)
df_timeseries.head()


In [ ]:
import plotly.graph_objects as go

region_names = {
    "Inden_CF": "Inden",
    "Weesow_Willmersdorf": "Weesow-Willmersdorf",
    "Lauingen" : "Lauingen",
    "Eggebek" : "Eggebek"
}

regions = df_timeseries["region"].unique()
n_rows = len(regions)

v_spacing = 0.06
row_height = (1 - v_spacing * (n_rows - 1)) / n_rows

def clip01(v):
    return max(0.0, min(1.0, v))

fig_soc = go.Figure()

for i, region in enumerate(regions):
    df_r = df_timeseries[df_timeseries["region"] == region]

    display_name = region_names.get(region, region)

    y_top = clip01(1 - i * (row_height + v_spacing))
    y_bottom = clip01(y_top - row_height)

    fig_soc.add_annotation(
    x=0.425,                 
    y=y_top + 0.02,         
    xref="paper",
    yref="paper",
    text=display_name,
    showarrow=False,
    font=dict(size=16))

    ax1, ax2, ax3 = 3 * i + 1, 3 * i + 2, 3 * i + 3
    xaxis_name = "x" if i == 0 else f"x{i+1}"
    y1_name = "y" if ax1 == 1 else f"y{ax1}"
    y2_name = f"y{ax2}"
    y3_name = f"y{ax3}"

    fig_soc.add_trace(go.Scatter(
        x=df_r["timestamp"], y=df_r["State_of_Charge_SOC_kWh"],
        name="SOC (kWh)", mode="lines", line=dict(color="blue"),
        xaxis=xaxis_name, yaxis=y1_name,
        legendgroup="soc", showlegend=(i == 0),
    ))

    fig_soc.add_trace(go.Scatter(
        x=df_r["timestamp"], y=df_r["PV_generation_kW"],
        name="power (kW)", mode="lines", line=dict(color="red"),
        xaxis=xaxis_name, yaxis=y2_name,
        legendgroup="power", showlegend=(i == 0),
    ))

    fig_soc.add_trace(go.Scatter(
        x=df_r["timestamp"], y=df_r["Spot_price_EUR_per_kWh"],
        name="price (EUR/kWh)", mode="lines", line=dict(color="green"),
        xaxis=xaxis_name, yaxis=y3_name,
        legendgroup="price", showlegend=(i == 0),
    ))

    fig_soc.update_layout({
        xaxis_name.replace("x", "xaxis"): dict(
            domain=[0, 0.85],
            anchor=y1_name,
            matches="x" if i > 0 else None,
            showticklabels=(i == n_rows - 1),
        ),
        y1_name.replace("y", "yaxis"): dict(
            title="SOC (kWh)", domain=[y_bottom, y_top],
            anchor=xaxis_name, color="blue",
        ),
        f"yaxis{ax2}": dict(
            title="power (kW)", domain=[y_bottom, y_top],
            anchor="free", overlaying=y1_name, side="right",
            position=0.85, color="red",
        ),
        f"yaxis{ax3}": dict(
            title="price (EUR/kWh)", domain=[y_bottom, y_top],
            anchor="free", overlaying=y1_name, side="right",
            position=0.93, color="green",
        ),
    })

fig_soc.update_layout(
    height=260 * n_rows,
    title="Annual Overview: SOC, PV Generation, and Spot Price by Region",
)

fig_soc.show()